## Objectives:
- Separation of tasks: tools, models, agentic frameworks
- Process of registering, testing, and integrating UC functions with LangChain using the `UCFunctionToolkit`
- Config and execute a LangChain agent with tool-calling capabilities.
- View and intepret the trace summary of agent execution and decision making.

## A. Setup and Inspect Data
- Requried: serverless version 4
- `databricks-langchain`

In [0]:
%pip install --upgrade databricks-langchain mlflow "unitycatalog-ai[databricks]==0.3.2" -qqq
%restart_python

In [0]:
catalog = "workspace"
schema = "bronze"
table_name = f"{catalog}.{schema}.sf_airbnb_listings"
avg_function_name = f"{catalog}.{schema}.avg_neigh_price"
cnt_function_name = f"{catalog}.{schema}.cnt_by_room_type"


def table_exists(table_name):
    catalog, schema, tbl = table_name.split('.')
    result = spark.sql(f"SHOW TABLES IN {catalog}.{schema} LIKE '{tbl}'")
    return result.count() > 0

# check if pre-requisite functions exist (see 01- notebook)
def function_exists(function_name):
    result = spark.sql(f"SHOW USER FUNCTIONS IN {catalog}.{schema} LIKE '{function_name.split('.')[-1]}'")
    return result.count() > 0
    
print(table_exists(table_name))
print(function_exists(avg_function_name))
print(function_exists(cnt_function_name))

In [0]:
df = spark.read.table(table_name)
display(df.limit(5))

## B. Init the Databricks Function Client

In [0]:
from unitycatalog.ai.core.databricks import DatabricksFunctionClient

# client = DatabricksFunctionClient() # for classic compute
client = DatabricksFunctionClient(execution_mode="serverless") # for serverless compute

## C. Agent Concepts - see training notes

## D. Integrate UC Functions with LangChain

### D1. Define the tools and toolkit

In [0]:
# UC fully-qualified function names
function_names = [avg_function_name, cnt_function_name]
print(f"Tool list: {function_names}")

In [0]:
# create the UC function toolkit
# this wraps the UC functions and makes them available as LangChain tools
from databricks_langchain import UCFunctionToolkit

toolkit = UCFunctionToolkit(function_names=function_names)
tools = toolkit.tools

In [0]:
tools[0]

### D2. Test the toolkit
- Execute example payloads using the toolkit defined by `tools` and the toolkit client by sending test payloads using the `execute_function` api
- Note: ensure the param:value in the payload complies to the function parameter names.

In [0]:
payload1 = {"neigh_name": "Mission"}
result1 = client.execute_function(
    function_name=tools[0].uc_function_name, parameters=payload1
)
print(result1.value)

In [0]:
payload2 = {
    "neigh_name": "Mission",
    "room_type_filter": "Private room"
}
result2 = client.execute_function(
    function_name=tools[1].uc_function_name, parameters=payload2
)
print(result2.value)

## E. Configure and Execute the Agent
The `AgentExecutor` method from `langchain.agents` acts as the orchestrator for the LLM to repeatedly invoke the agent's decision function, handle tool execution, and manage the flow of information between the reasoning, action, and observation steps.

In [0]:
import json

with open("./demo_agent.json", "r") as f:
    config = json.load(f)
llm_endpoint = config["llm_endpoint"]
llm_temperature = config["llm_temperature"]
system_prompt = config["system_prompt"]

print(f"llm_endpoint: {llm_endpoint}")
print(f"llm_temperature: {llm_temperature}")
print(f"system_prompt: {system_prompt}")

In [0]:
# from langchain.agents import AgentExecutor, create_tool_calling_agent # depreciated from langchain v1.0+
from langchain.agents import create_agent # use this instead

# from langchain.prompts import ChatPromptTemplate # depreciated
from langchain_core.prompts import ChatPromptTemplate # use this instead

from databricks_langchain import ChatDatabricks
import mlflow


mlflow.langchain.autolog()

llm = ChatDatabricks(
    endpoint = llm_endpoint,
    temperature = llm_temperature
)


### [Depreciated in new LangChain] Using `AgentExecutor`

In [0]:
# prompt_payload = ChatPromptTemplate.from_messages(
#     [
#         ("system", system_prompt),
#         ("placeholder", "{chat_history}"),
#         ("human", "{input}"),
#         ("placeholder", "{agent_scratchpad}") # for agent to reason and plan
#     ]
# )

# agent_config = create_tool_calling_agent(
#     llm,
#     tools,
#     prompt_payload
# )

# agent_executor = AgentExecutor(agent=agent_config, tools=tools, verbose=True)
# response = agent_executor.invoke(
#     {
#         "input": "Get the average price for Mission and tell me the number of properties there that have a shared room"
#     }
# )

### New React Agent Interface
`create_agent` handles the reasoning loop (ReAct) under the hood. chat_history and agent_scratchpad are managed internally.

In [0]:
# Use create_agent directly (replaces create_tool_calling_agent and AgentExecutor)
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt # Pass system prompt directly
)

# 3. Invoke with a list of messages
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "Get the average price for Mission and tell me the number of properties there that have a shared room"}
    ]
})

In [0]:
# the ai response is always the final message
print(response['messages'][-1].content)